# Reserve 1: Saros-Grid-SSM - Google Colab Runner
**Project:** Solar/Wind Spatio-Temporal SSM for Spanish REE Electricity Grid  
**Author:** Jaime Jarauta  
**Pre-Registered Question:** Can continuous-time selective state space recurrence coupled with graph spatial diffusion predict 15-minute nodal renewable injections with wMAPE < 6.5% and curtailment F1 >= 0.75 without $O(N^2)$ attention?

Configured for Google Colab (Free T4 or Pro A100/L4) and local GPUs.


In [ ]:
# Step 1: Pre-flight Hardware Verification
import sys, os, time
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Device: {dev_name} | VRAM: {vram_gb:.2f} GB")
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = True
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Device: CPU")


In [ ]:
# Step 2: Dependencies & Imports
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q uv
    !uv pip install --system -q pyyaml wandb pytest

if "src" not in sys.path:
    sys.path.insert(0, "src")

from saros_grid_ssm.model import GridSpatioTemporalSSM
from saros_grid_ssm.ree_data import generate_synthetic_ree_grid, REESpanishGridDataset
from saros_grid_ssm.metrics import compute_wmape, compute_curtailment_f1, compute_grid_metrics

print("Saros-Grid-SSM modules loaded successfully.")


In [ ]:
# Step 3: Generate REE 50-Node Grid & Initialize Model
seed = 2026
torch.manual_seed(seed)
num_nodes = 50

raw_data = generate_synthetic_ree_grid(num_days=21, num_nodes=num_nodes, seed=seed)
adj_norm = torch.from_numpy(raw_data["adjacency_norm"]).float().to(device)

train_split = 14 * 96
train_dataset = REESpanishGridDataset(
    inputs=raw_data["inputs"][:train_split],
    injections=raw_data["injections"][:train_split],
    curtailments=raw_data["curtailments"][:train_split],
    window_size=96,
    stride=48
)
val_dataset = REESpanishGridDataset(
    inputs=raw_data["inputs"][train_split:],
    injections=raw_data["injections"][train_split:],
    curtailments=raw_data["curtailments"][train_split:],
    window_size=96,
    stride=96
)

from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

model = GridSpatioTemporalSSM(
    in_features=4,
    out_features=2,
    d_model=64,
    d_state=16,
    num_spatial_hops=2,
    num_layers=2,
    bidirectional=True,
    dropout=0.05
).to(device)

print(f"Initialized GridSpatioTemporalSSM ({sum(p.numel() for p in model.parameters()):,} params)")


In [ ]:
# Step 4: Run Smoke Training Loop (15-min dispatch)
import torch.nn as nn

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
huber_loss_fn = nn.SmoothL1Loss(beta=10.0)
bce_loss_fn = nn.BCELoss()

model.train()
steps, max_steps = 0, 50
t0 = time.time()

while steps < max_steps:
    for x_b, y_inj_b, y_curt_b in train_loader:
        x_b, y_inj_b, y_curt_b = x_b.to(device), y_inj_b.to(device), y_curt_b.to(device)
        optimizer.zero_grad()
        preds = model(x_b, adj_norm)
        loss = huber_loss_fn(preds["injection_mw"], y_inj_b) + 5.0 * bce_loss_fn(preds["curtailment_prob"], y_curt_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        steps += 1
        if steps % 10 == 0:
            wmape_step = compute_wmape(y_inj_b, preds["injection_mw"])
            print(f"Step {steps:02d}/{max_steps} | Loss: {loss.item():.4f} | Batch wMAPE: {wmape_step * 100:.2f}%")
        if steps >= max_steps:
            break

print(f"Training elapsed: {time.time() - t0:.2f}s")


In [ ]:
# Step 5: Evaluate Pre-Registered Gate Criteria
model.eval()
with torch.no_grad():
    for x_val, y_inj_val, y_curt_val in val_loader:
        x_val, y_inj_val, y_curt_val = x_val.to(device), y_inj_val.to(device), y_curt_val.to(device)
        preds_val = model(x_val, adj_norm)
        metrics = compute_grid_metrics(
            y_inj_val, preds_val["injection_mw"],
            y_curt_val, preds_val["curtailment_prob"]
        )

wmape_pct = metrics["wmape"] * 100.0
gate_passed = metrics["wmape"] < 0.065 and metrics["curtailment_f1"] >= 0.75

print("=" * 60)
print(f"Day-Ahead Injection wMAPE : {wmape_pct:.2f}% (Target: < 6.50%)")
print(f"Curtailment F1 Score      : {metrics['curtailment_f1']:.4f} (Target: >= 0.750)")
print(f"Pre-Registered Gate 1     : {'PASSED' if gate_passed else 'FAILED'}")
print("=" * 60)


In [ ]:
# Step 5: Credit Protection & Automatic Machine Teardown
# @title Auto-Kill Runtime on Completion to Save Colab Credits
AUTO_KILL_ON_FINISH = True # @param {type:"boolean"}

if AUTO_KILL_ON_FINISH:
    print("\n[Credit Guard] Run complete. Unassigning Colab GPU immediately to prevent idle credit burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"(Auto-kill skipped: not running inside interactive Colab: {e})")
